<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.1-token-economics/practice/GCP_Capstone_2.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 2.1 — Token Economics

8 hands-on exercises with complete solutions. Master tokenization, multilingual costs, context budgeting, and build a reusable token economics module.

Runnable companion to the published practice lab. Each exercise below shows the objective and a complete solution. Cloud Shell / `gcloud` steps are `%%bash` cells; Python steps run in Colab after you authenticate and set your project.

---

## Exercise 1: Tokenize 10 Texts with LocalTokenizer  
**Difficulty:** Easy

Use the offline LocalTokenizer to tokenize 10 different text types and discover the chars-per-token ratio.

1. Install google-genai if needed
2. Create LocalTokenizer for gemini-3.6-flash
3. Tokenize: short word, sentence, paragraph, code snippet, URL, JSON, number, Hindi, Telugu, mixed

**Solution:**

In [ ]:
# --- Setup: install + auth (run me first) ---
!pip install -q "google-genai[local-tokenizer]"
from google.colab import auth
auth.authenticate_user()

from google import genai
PROJECT_ID = 'documind-ai-YOUR-ID'   # CHANGE THIS to your project id
client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # global: Gemini 3.x generation/count_tokens
print('Client ready')

In [ ]:
from google.genai.local_tokenizer import LocalTokenizer

tokenizer = LocalTokenizer(model_name="gemini-3-flash-preview")

texts = [
    "Hello",
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is a subset of artificial intelligence." * 5,
    "def hello():\n    print('Hello World')",
    "https://cloud.google.com/vertex-ai/docs/start",
    '{"name": "Sart", "city": "Hyderabad", "age": 28}',
    "3.14159265358979323846",
    "नमस्ते दुनिया",
    "నమస్తె ప్రపంచం",
    "Hello दुनिया world ప్రపంచం",
]

print(f"  {'Type':<15} {'Chars':>6} {'Tokens':>7} {'Ratio':>7}")
for t in texts:
    r = tokenizer.count_tokens(t)
    ratio = len(t) / r.total_tokens
    label = t[:15].replace("\n"," ")
    print(f"  {label:<15} {len(t):>6} {r.total_tokens:>7} {ratio:>5.1f}:1")

## Exercise 2: Hindi/Telugu Token Tax Calculator  
**Difficulty:** Easy

Compare the same semantic content in English, Hindi, and Telugu. Calculate the exact cost multiplier.

1. Write equivalent sentences in all 3 languages
2. Count tokens for each using API
3. Calculate multiplier vs English
4. Estimate cost impact for 10K queries/day

**Solution:**

In [ ]:
sentences = {
    "English": "Artificial intelligence is changing how we work and live.",
    "Hindi": "कृत्रिम बुद्धिमत्ता हमारे काम और जीवन के तरीके को बदल रही है।",
    "Telugu": "ఆర్టిఫిషియల్ ఇంటెలిజెన్స్ మన పని మరియు జీవన విధానాలను మార్చుతోంది.",
}

en_tok = None
print(f"  {'Lang':<10} {'Tokens':>7} {'Multiplier':>11} {'Cost/10K queries':>17}")
for lang, text in sentences.items():
    r = client.models.count_tokens(model="gemini-3.6-flash", contents=text)
    if lang == "English": en_tok = r.total_tokens
    mult = r.total_tokens / en_tok
    # Cost for 10K queries at Flash rates (input only)
    cost_inr = 10000 * r.total_tokens * 1.50 / 1e6 * 85
    print(f"  {lang:<10} {r.total_tokens:>7} {mult:>9.1f}x    ₹{cost_inr:>13.2f}")

## Exercise 3: Multimodal Token Counter  
**Difficulty:** Easy

Count tokens for different modalities: images at different resolutions, audio, and PDF pages.

1. Upload a sample image
2. Count tokens at low, medium, high resolution
3. Estimate audio tokens for 1-minute clip
4. Calculate PDF token cost for a 20-page document

**Solution:**

In [ ]:
# Image token costs by resolution
image_costs = {
    "Gemini 3 LOW": 280,
    "Gemini 3 MEDIUM": 560,
    "Gemini 3 HIGH": 1120,
    "Gemini 3 ULTRA_HIGH": 2240,
}

print("🖼 Image Token Costs:")
for name, tokens in image_costs.items():
    cost = tokens * 1.50 / 1e6 * 85
    print(f"  {name:<25} {tokens:>6} tokens  ₹{cost:.4f}/image")

# Audio: 32 tokens/second
audio_secs = [10, 60, 300, 3600]
print("\n🎧 Audio Token Costs (32 tok/sec):")
for s in audio_secs:
    tokens = s * 32
    cost = tokens * 1.50 / 1e6 * 85  # Audio rate = $1.50/M
    label = f"{s}s" if s < 60 else f"{s//60}min"
    print(f"  {label:<10} {tokens:>8,} tokens  ₹{cost:.4f}")

# PDF: ~258-560 tokens per page
pages = 20
tokens_per_page = 258  # Gemini 3.x
total = pages * tokens_per_page
cost = total * 1.50 / 1e6 * 85
print(f"\n📄 20-page PDF: {total:,} tokens | ₹{cost:.4f}")

## Exercise 4: Context Window Budget Calculator  
**Difficulty:** Medium

Build the budget_context() function. Test with a realistic DocuMind scenario.

1. Define function with system, history, rag, query components
2. Use count_tokens for each
3. Calculate utilization percentage
4. Test with 10-turn conversation + RAG context

**Solution:**

In [ ]:
def budget_context(client, model, system="", history=None,
                    rag_context="", query="", margin=200):
    info = client.models.get(model=model)
    max_in = info.input_token_limit or 1_048_576
    max_out = info.output_token_limit or 65_536

    parts = {}
    for name, txt in [("system",system),("rag",rag_context),("query",query)]:
        parts[name] = client.models.count_tokens(
            model=model, contents=txt
        ).total_tokens if txt else 0
    parts["history"] = client.models.count_tokens(
        model=model, contents=history
    ).total_tokens if history else 0

    total = sum(parts.values()) + margin
    avail = min(max_out, max_in - total)
    pct = total / max_in * 100

    print(f"📊 Budget for {model}:")
    for k, v in parts.items():
        print(f"  {k:<12} {v:>8,} tokens")
    print(f"  {'margin':<12} {margin:>8,} tokens")
    print(f"  {'TOTAL':<12} {total:>8,} tokens ({pct:.2f}%)")
    print(f"  {'Available':<12} {avail:>8,} for output")
    return {"total": total, "available": avail}

# Test with DocuMind scenario
budget_context(
    client, "gemini-3.6-flash",
    system="You are DocuMind AI. Cite sources. Be concise.",
    rag_context="[Document chunk 1] ... [Document chunk 5]" * 100,
    query="Summarize the key findings on transformer architectures.",
)

## Exercise 5: 6-Model Cost Comparison Table  
**Difficulty:** Medium

Run the same prompt on 3 models. Build a comparison table with tokens, latency, and INR cost.

1. Define prompt and 5 model configs
2. Time each call
3. Extract usage_metadata
4. Build formatted table with cost per 1000 queries

**Solution:**

In [ ]:
import time

prompt = "Explain the CAP theorem in distributed systems. Give a real-world example."
models = [
    ("gemini-3.1-flash-lite", 0.25, 1.50),
    ("gemini-3.6-flash", 1.50, 7.50),
    ("gemini-3.1-pro-preview", 2.00, 12.00),
]

print(f"{'Model':<38} {'Tok':>5} {'ms':>6} {'Rs/call':>8} {'Rs/1K':>8}")
print("-"*70)
for name,ip,op in models:
    t0=time.time()
    try:
        r=client.models.generate_content(model=name,contents=prompt)
        ms=(time.time()-t0)*1000
        u=r.usage_metadata
        cost=(u.prompt_token_count*ip+(u.candidates_token_count+(getattr(u,'thoughts_token_count',0) or 0))*op)/1e6*85
        print(f"{name:<38} {u.total_token_count:>5} {ms:>5.0f} Rs{cost:>7.4f} Rs{cost*1000:>7.2f}")
    except Exception as e:
        print(f"{name:<38} ERROR: {e}")

## Exercise 6: Multi-Turn Cost Growth Simulator  
**Difficulty:** Medium

Simulate a 20-turn conversation. Plot cumulative cost growth. Find where a sliding window delivers significant savings.

1. Simulate token accumulation over 20 turns
2. Calculate per-turn and cumulative cost
3. Compare with 5-turn sliding window
4. Find the crossover point where windowing savings become significant

**Solution:**

In [ ]:
sys_tok = 200
tok_per_turn = 400
out_per_turn = 200
INR = 85

print("📈 Multi-Turn Cost: Full History vs 5-Turn Window")
print(f"  {'Turn':>4} {'Full Rs':>10} {'Window Rs':>10} {'Savings':>9}")

cum_full = 0
cum_win = 0
for turn in range(1, 21):
    # Full history
    full_input = sys_tok + (turn-1)*tok_per_turn + 200
    full_cost = (full_input*1.50 + out_per_turn*7.50)/1e6*INR
    cum_full += full_cost

    # 5-turn sliding window
    win_history = min(turn-1, 5) * tok_per_turn
    win_input = sys_tok + win_history + 200
    win_cost = (win_input*1.50 + out_per_turn*7.50)/1e6*INR
    cum_win += win_cost

    savings = (cum_full - cum_win) / cum_full * 100 if cum_full > 0 else 0
    print(f"  {turn:>4} Rs{cum_full:>9.4f} Rs{cum_win:>9.4f}  {savings:>6.1f}%")

print(f"\n  Total saved by windowing: Rs{cum_full-cum_win:.4f} ({(cum_full-cum_win)/cum_full*100:.0f}%)")

## Exercise 7: Multilingual Cost Optimizer  
**Difficulty:** Challenge

Build a function that detects input language, estimates the token multiplier, and recommends the cheapest model+tier.

1. Detect language using a simple heuristic or Gemini classification
2. Apply multiplier: English 1x, Hindi 2x, Telugu 2.2x
3. Calculate cost across all models
4. Recommend cheapest model that meets quality threshold

**Solution:**

In [ ]:
LANG_MULTIPLIERS = {"english":1.0, "hindi":2.0, "telugu":2.2, "tamil":2.1, "bengali":2.0}
PRICING = {
    "flash-lite": (0.25,1.50), "flash": (1.50,7.50),
    "pro": (2.00,12.00),
}

def optimize_cost(text, lang="english", avg_output=200, budget_inr=0.05):
    mult = LANG_MULTIPLIERS.get(lang, 1.5)
    base_tokens = client.models.count_tokens(
        model="gemini-3.6-flash", contents=text
    ).total_tokens
    est_tokens = int(base_tokens * mult)  # adjust for language

    print(f"\n📊 Language: {lang} (multiplier: {mult}x)")
    print(f"   Base tokens: {base_tokens} | Estimated: {est_tokens}")
    print(f"   Budget: Rs {budget_inr}/query\n")

    for model, (ip, op) in PRICING.items():
        cost = (est_tokens*ip + avg_output*op)/1e6*85
        fits = "✅" if cost <= budget_inr else "❌"
        print(f"   {fits} {model:<12} Rs {cost:.4f}/query")

optimize_cost("कृत्रिम बुद्धिमत्ता क्या है?", lang="hindi", budget_inr=0.02)

## Exercise 8: Complete Token Economics Module  
**Difficulty:** Challenge

Build the production-ready token_economics.py module with all counting methods, pricing, and projections.

1. Define PRICING dict for all 3 models
2. Implement count_local() with LocalTokenizer
3. Implement cost_inr() reading usage_metadata
4. Implement monthly_projection() with model routing mix
5. Test all functions with DocuMind scenarios

**Solution:**

In [ ]:
"""Token Economics Module for DocuMind AI Capstone"""
from google.genai.local_tokenizer import LocalTokenizer

PRICING = {
    "gemini-3.1-flash-lite": {"input":0.25,"output":1.50,"cached":0.025,"batch_in":0.125,"batch_out":0.75},
    "gemini-3.6-flash":      {"input":1.50,"output":7.50,"cached":0.15,"batch_in":0.75,"batch_out":3.75},
    "gemini-3.1-pro-preview":        {"input":2.00,"output":12.0,"cached":0.20,"batch_in":1.00,"batch_out":6.0},
}
INR = 85
_tok = LocalTokenizer(model_name="gemini-3-flash-preview")

def count_local(text): return _tok.count_tokens(text).total_tokens

def cost_inr(meta, model="gemini-3.6-flash"):
    p=PRICING.get(model,PRICING["gemini-3.6-flash"])
    think=(getattr(meta,"thoughts_token_count",0) or 0)
    i=meta.prompt_token_count*p["input"]/1e6
    o=(meta.candidates_token_count+think)*p["output"]/1e6
    return {"inr":round((i+o)*INR,4),"usd":round(i+o,6)}

def monthly_projection(qpd, avg_in, avg_out,
                       mix={"gemini-3.1-flash-lite":0.6,"gemini-3.6-flash":0.3,"gemini-3.1-pro-preview":0.1}):
    monthly=qpd*30; total=0
    for model,pct in mix.items():
        p=PRICING[model]; q=monthly*pct
        total+=(q*avg_in*p["input"]+q*avg_out*p["output"])/1e6
    return {"usd":round(total,2),"inr":round(total*INR,2),"months":round(500/total,1) if total>0 else float("inf")}

# Test
print("Local count:", count_local("Hello Hyderabad!"))
r = monthly_projection(100, 2000, 500)
print(f"Monthly: ${r['usd']}/mo (Rs {r['inr']}) | $500 lasts {r['months']} months")